# Exercise 6 — Manufacturing Translation

In this exercise we take the control concepts from previous exercises and see them in three real manufacturing systems.  
For each system we identify the control loop elements and simulate the closed-loop response.

## Section 1 — Conveyor Belt Speed Control

| Element | Description |
|---|---|
| **Controlled variable** | Belt speed [m/s] |
| **Manipulated variable** | Motor voltage command |
| **Disturbance** | Varying box / product load on the belt |
| **Sensor** | Shaft encoder or tachometer |
| **Poor tuning consequence** | Boxes pile up, spacing errors, downstream jam |

In [1]:
import numpy as np
from mfg_helpers import (
    default_conveyor_params, simulate_conveyor, build_mfg_figure,
    default_oven_params, simulate_oven,
    default_tank_params, simulate_tank,
)

In [2]:
params = default_conveyor_params()
t = np.arange(0, params["t_end"], params["dt"])
target = np.full(len(t), 1.5)  # target speed [m/s]

# Disturbance: heavy boxes hit belt at t=15s, removed at t=35s
load = np.zeros(len(t))
load[(t >= 15) & (t < 35)] = 30.0  # extra friction [N]

result = simulate_conveyor(params, target, load)

labels = dict(y_label="Belt Speed", u_label="Motor Cmd", dist_label="Box Load",
              y_unit="m/s", u_unit="unit", dist_unit="N")
fig = build_mfg_figure(result, labels, title="Conveyor Belt — Load Disturbance")
fig.show()

### Instructor Note

Notice how the PID controller recovers belt speed after the load change. The dip when boxes arrive is the disturbance response. On a real line, that dip means product spacing errors — downstream equipment may jam or starve.

## Section 2 — Oven / Furnace Temperature Control

| Element | Description |
|---|---|
| **Controlled variable** | Chamber temperature [°C] |
| **Manipulated variable** | Heater power command |
| **Disturbance** | Door open events, part loading (heat sink), ambient changes |
| **Sensor** | Thermocouple |
| **Poor tuning consequence** | Under-cured parts (too cold), scorched parts (overshoot), energy waste |

In [3]:
params = default_oven_params()
t = np.arange(0, params["t_end"], params["dt"])
target = np.full(len(t), 200.0)  # target temperature [°C]

# Disturbance: door open at t=80s, closed at t=140s
heat_loss = np.zeros(len(t))
heat_loss[(t >= 80) & (t < 140)] = 15.0  # extra heat loss [°C]

result = simulate_oven(params, target, heat_loss)

labels = dict(y_label="Temperature", u_label="Heater Power", dist_label="Heat Loss",
              y_unit="°C", u_unit="unit", dist_unit="°C")
fig = build_mfg_figure(result, labels, title="Oven — Door-Open Disturbance")
fig.show()

### Instructor Note

The thermal time constant (tau=30s) means temperature drops slowly after the door opens — this is typical of ovens. Watch how the heater saturates trying to compensate. In production, this would violate recipe hold times and risk scrapping the batch.

## Section 3 — Tank Level Control

| Element | Description |
|---|---|
| **Controlled variable** | Liquid level [m] |
| **Manipulated variable** | Pump speed / valve opening |
| **Disturbance** | Downstream demand changes (variable outflow) |
| **Sensor** | Level transmitter (pressure or ultrasonic) |
| **Poor tuning consequence** | Overflow (safety hazard), starved downstream process, pump cavitation |

In [4]:
params = default_tank_params()
t = np.arange(0, params["t_end"], params["dt"])
target = np.full(len(t), 1.0)  # target level [m]

# Disturbance: downstream process draws extra at t=30s
outflow = np.zeros(len(t))
outflow[(t >= 30) & (t < 70)] = 0.005  # extra outflow [m^3/s]

result = simulate_tank(params, target, outflow)

labels = dict(y_label="Level", u_label="Pump Cmd", dist_label="Extra Outflow",
              y_unit="m", u_unit="unit", dist_unit="m\u00b3/s")
fig = build_mfg_figure(result, labels, title="Tank Level — Variable Outflow")
fig.show()

### Instructor Note

The tank is nonlinear — outflow depends on sqrt(h). This means the system behaves differently at high vs low levels. Notice the asymmetric response: draining is faster than filling because gravity helps.

## Section 4 — Comparison Summary

| System | Controlled Var | Manipulated Var | Key Disturbance | Production Risk |
|---|---|---|---|---|
| Conveyor | Belt speed | Motor voltage | Box loading | Spacing / jam |
| Oven | Temperature | Heater power | Door open | Scrap / under-cure |
| Tank | Liquid level | Pump command | Demand change | Overflow / starvation |

All three systems share the same PID structure. The difference is in the physics, the time constants, and what "failure" looks like on the plant floor.

### Student Challenge

Pick one of the three systems above. Modify the PID gains to be deliberately poor (e.g. set `Ki=0` or make `Kp` very large). Re-run the simulation and describe in 2-3 sentences what would happen in production.

In [5]:
# ---- Student challenge: modify gains and re-run ----
# Example: params["Kp"] = 20.0  # aggressive P gain
# Re-run the simulation and observe the production impact